## Step 1: Frame sorting

### Overview

The **PL data reduction process** begins with frame sorting — organizing PL frames based on the PSF peak coordinates. 

**Tutorial Series**:
- [Step 2: Spectral extraction](../visPLred/tutorials/step2_spectral_extraction.ipynb) 
- [Step 3: Image reconstruction](step3_image_reconstruction.ipynb) 

---

### Process Workflow

The frame sorting procedure involves four key stages:

1. **Timestamp Matching**  
   Match timestamps between the two cameras (The PSF camera usually runs faster than the PL camera.)

2. **PSF Peak Detection**  
   Identify PSF peaks within each frame

3. **Grid Definition**  
   Define a grid to sort the PL frames to

4. **Frame Sorting**  
   Sort PL frames into the defined grid and save them to a file

---


> **Note:** The examples in this notebook use a short dataset for demonstration purposes (~50 frames of PL data, 1 second observation). The PL data are in `data/slowcam/` and the PSF data are in `data/fastcam/`.

### 1. Timestamp matching

First we match timestamps (txt files in `data/fastcam` and `data/slowcam`). The slowcam will be used as a reference, and the fastcam timestamps and frames will be matched to the slowcam timestamps.

In [ ]:
import sys
sys.path.insert(0, '../..')
import PLred.sort as sort
import matplotlib.pyplot as plt
import numpy as np

The below script generates three files:

1. **_fastcam_matched_frames.npy**  
   The PSF frames whose timestamps were matched to the PL frames
2. **.pkl**  
   Dictionary of matched file/frame indices.
3. **_nstacks.npy**  
   Number of stacked fastcam frames per slowcam frame

In [ ]:
%cd data/
configname = 'timestamp_matching_config.ini'
sort.script_match_timestamps(configname)
%cd ..

In [ ]:
import h5py
with h5py.File('timestamp_matching_output/first_palila_matched.h5', 'r') as f:
    print(list(f.keys()))
    frames = f['frames'][:]
    timestamps = f['timestamps'][:]
    

In [ ]:
timestamps[0]

In [ ]:
import PLred.ingest as ingest

h5_giant = ingest.ingest_to_h5(
    step1_h5       = 'timestamp_matching_output/first_palila_matched.h5',
    plcam_dark     = 'data/slowcam/dark.fits',
    psfcam_dark    = 'data/fastcam/dark.fits',
    outpath        = 'timestamp_matching_output/test.h5',
    plcam_roi      = (0,412,1200,1220),
    plcam_data_dir = 'data/slowcam/',
    verbose        = True,
)

In [ ]:
import h5py

with h5py.File('timestamp_matching_output/test.h5', 'r') as f:
    print(f['metadata'].keys())
    print('attrs:', dict(f.attrs))
    print('psfcam/frames   :', f['psfcam/frames'].shape, f['psfcam/frames'].dtype)
    print('psfcam/centroids:', f['psfcam/centroids'].shape)
    print('psfcam/peaks    :', f['psfcam/peaks'][:5])
    print('plcam/frames    :', f['plcam/frames'].shape, f['plcam/frames'].dtype)
    print('timestamps      :', f['metadata/timestamps'].shape)

In [ ]:
with h5py.File('timestamp_matching_output/test.h5', 'r') as f:
    psfframes  = f['psfcam/frames'][:]
    centroids  = f['psfcam/centroids'][:]
    peaks      = f['psfcam/peaks'][:]
    plcam_frame = f['plcam/frames'][0]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(psfframes[0], origin='upper')
axes[0].plot(centroids[0, 0], centroids[0, 1], 'r+', ms=10)
axes[0].set_title('PSFcam frame 0  (peak=%.0f)' % peaks[0])

axes[1].scatter(centroids[:, 0], centroids[:, 1], c=peaks, s=20)
axes[1].set_xlabel('x (px)'); axes[1].set_ylabel('y (px)')
axes[1].set_title('PSF centroids (colour = peak / Strehl)')
plt.colorbar(axes[1].collections[0], ax=axes[1])

axes[2].imshow(plcam_frame, origin='upper', aspect='auto')
axes[2].set_title('PLcam frame 0 (dark-subtracted)')

plt.tight_layout()
plt.show()

# add here

### Step 3: Grid exploration and averaging

With the giant H5 from Step 2 in hand, we use `PLred.average` to:

1. **`explore_grid()`** — tune grid resolution, FOV, and Strehl / time filters interactively; optionally preview response maps at a few PLcam pixels
2. **`average_to_h5()`** — write a compact averaged H5 once the parameters look good
3. **`pixel_map()`** — extract a 2-D response map from the averaged H5 for any pixel

In [ ]:
import PLred.average as average
import importlib
importlib.reload(average)
import PLred.average as average

import h5py

giant_h5 = 'timestamp_matching_output/test.h5'

#### 3a. Build ROI access cache (optional but recommended)

`build_ROI_access()` adds a transposed dataset `plcam/roi_access (roi_h, roi_w, N)` with `chunks=(1, 1, chunk_t)` into the **same** giant H5. This is the fast path for `explore_grid()`:

| Layout | Chunks read per pixel | Use case |
|---|---|---|
| `plcam/frames (N, ny, nx)` chunks=(1,ny,nx) | N chunks × ny×nx values | frame-wise averaging |
| `plcam/roi_access (roi_h, roi_w, N)` chunks=(1,1,chunk_t) | ≈N/chunk_t chunks × chunk_t values | pixel time-series exploration |

Choose an `roi` that covers the pixels you plan to inspect. Pass `zarr_path` to also write a Zarr store for the HTML viewer later.

In [ ]:
# roi = (y0, y1, x0, x1) in PLcam pixel coordinates (after ROI crop in Step 2)
# Here the stored PLcam frames are (100, 200) so valid range is y:[0,100), x:[0,200)
roi = (0, 412, 0, 10)   # cover the full stored frame for this tutorial

average.build_ROI_access(
    giant_h5,
    roi       = roi,
    chunk_t   = 2048,    # time-axis chunk size; ceil(N/chunk_t) reads per pixel
    overwrite = False,   # skip if already built
    # zarr_path = 'timestamp_matching_output/roi_access.zarr',  # uncomment for HTML viewer
    verbose   = True,
    zarr_path="timestamp_matching_output/roi_access.zarr",
    zarr_include_metadata=True,
    zarr_include_psf= True,
    h5_path = 'timestamp_matching_output/roi.h5',
)

In [ ]:
# Verify roi_access was added to the giant H5
with h5py.File(giant_h5, 'r') as f:
    ra = f['plcam/roi_access']
    print('roi_access shape :', ra.shape)    # (roi_h, roi_w, N)
    print('roi_access chunks:', ra.chunks)   # (1, 1, chunk_t)
    print('roi attr         :', list(ra.attrs['roi']))
    # # Spot-check: roi_access[py, px, :] == frames[:, py, px]
    # py, px = 43, 100
    # fast = ra[py, px, :]
    # slow = f['plcam/frames'][:, py, px]
    # print('max diff (fast vs slow):', abs(fast - slow).max())

#### 3b. Explore grid coverage (no PLcam read)

`explore_grid()` with no `plcam_pixels` reads only centroids, peaks, and timestamps — very fast regardless of dataset size. Use this to tune `map_n` and `map_width` until the nframes map looks well-filled.

In [ ]:
centroids, peaks, timestamps, plcam_type, t0 = average._load_giant_meta(giant_h5)


In [ ]:
timestamps